# exp097_modelpkg_tiny_gate_on_exp073 train

exp073 inference prediction と model-package-only prediction を align し、tiny gate grid の diff guard を保存する。学習は行わない。

## Contents

1. Setup and configuration
2. Input alignment
3. Gate grid audit
4. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, load_config
from modelpkg_tiny_gate_on_exp073 import load_and_align, run_audit_from_config

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print('Experiment:', EXPERIMENT_NAME)
print('Route:', config['experiment']['route'])
print('Status:', config['experiment']['status'])
print('Sample submission:', paths.sample_submission_path)
print('Artifacts:', paths.artifacts_dir)
print('Selected mode/model:', config['audit']['selected_mode'], config['audit']['selected_model'])
print('Gate grid:', config['audit']['grid'])
print('Selected variant:', config['inference']['selected_variant'])

## 2. Input alignment

The base prediction is exp073 `lgb_mean`. The correction source is `submission_model_package_only.csv`; both are aligned to `sample_submission.csv` order.

In [ ]:
aligned, input_meta = load_and_align(config, sample_path=paths.sample_submission_path)
print(json.dumps(input_meta, indent=2)[:4000])
display(aligned[['id', 'well', 'base_tvt', 'modelpkg_tvt', 'diff_modelpkg_minus_base', 'abs_modelpkg_diff']].head())
display(aligned['abs_modelpkg_diff'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame('abs_modelpkg_diff'))

## 3. Gate grid audit

Each candidate applies the same agreement gate with different `gmax` and `scale`, then records correction magnitude and SHA values.

In [ ]:
summary = run_audit_from_config(
    config,
    sample_path=paths.sample_submission_path,
    output_dir=paths.artifacts_dir,
)
print(json.dumps(summary['selected_summary'], indent=2))
print('Selected passes all guards:', summary['selected_passes_all_guards'])

## 4. Metrics and artifacts


In [ ]:
metrics = pd.read_csv(summary['artifacts']['variant_metrics'])
display(metrics)

run_metrics = {
    'experiment': EXPERIMENT_NAME,
    'status': summary['status'],
    'route': config['experiment']['route'],
    'cv': None,
    'public_lb': None,
    'private_lb': None,
    'selected_variant': summary['selected_variant'],
    'selected_passes_all_guards': summary['selected_passes_all_guards'],
    'selected_summary': summary['selected_summary'],
    'artifacts': summary['artifacts'],
}
paths.metrics_path.write_text(json.dumps(run_metrics, indent=2, sort_keys=True) + '\n')
print('Metrics written:', paths.metrics_path)